# Lot D · Silver · étape 1 : fiche par LLM (Qwen2.5-7B-Instruct, 4 bits)

Exécution sur Colab (Exécution → Modifier le type d'exécution → **GPU T4**). Dev uniquement : le test gelé n'est jamais lu.
Toute mise au point se fait sur le dev ; chaque exécution retenue est consignée dans `experiments.md`.

In [ ]:
!nvidia-smi -L

## 1. Récupérer la branche `lot-d-silver`
Cellule ré-exécutable : elle clone une seule fois dans `/content`, puis met à jour avec `git pull`.

In [ ]:
%cd /content
import os
if os.path.isdir('fardi-caseforge/.git'):
    !git -C fardi-caseforge pull -q origin lot-d-silver
else:
    !git clone -q -b lot-d-silver https://github.com/onsalouini/fardi-caseforge.git
%cd /content/fardi-caseforge
!git log --oneline -1

## 2. Dépendances (torch de Colab conservé)

In [ ]:
!pip install -q "transformers>=4.44" accelerate bitsandbytes

## 3. Essai rapide sur 2 documents du dev
Vérifie le chargement du modèle, le JSON et le journal (temps par document, éléments retirés, repli).

In [ ]:
!python generation/llm.py --split dev --graine 42 --limite 2 --out generation/sorties/essai.jsonl

## 4. Dev complet, graine 42, configuration complète (exemples=3, entités, contrôle)

In [ ]:
!python generation/llm.py --split dev --graine 42 --out generation/sorties/llm_dev_s42.jsonl

## 5. Évaluation (même évaluateur que le gabarit) et comparaison au gabarit v2

In [ ]:
!python generation/gabarit.py --split dev --out generation/sorties/gabarit_v2_dev.jsonl
!python generation/evaluer.py --gen generation/sorties/gabarit_v2_dev.jsonl --out generation/sorties/eval_gabarit_v2.md > /dev/null
!test -s generation/sorties/llm_dev_s42.jsonl || echo 'ERREUR : la génération (section 4) n a produit aucune fiche'
!python generation/evaluer.py --gen generation/sorties/llm_dev_s42.jsonl --out generation/sorties/eval_llm_s42.md

## 6. Résumé du journal des contrôles

In [ ]:
import json
j = [json.loads(l) for l in open('generation/sorties/llm_dev_s42.journal.jsonl', encoding='utf-8')]
print('replis gabarit :', sum(x['repli_gabarit'] for x in j))
print('essais > 1     :', sum(x['essais'] > 1 for x in j))
print('retirés (non ancrés) :', sum(len(x.get('retires_non_ancres', [])) for x in j))
print('citations corrigées  :', sum(len(x.get('citations_corrigees', [])) for x in j))
print('retirés (sensibles)  :', sum(len(x.get('retires_sensibles', [])) for x in j))
print('temps moyen / doc (s):', round(sum(x['temps_s'] for x in j) / len(j), 1))

## 7. Télécharger les sorties (à remettre dans `generation/sorties/` en local)

In [ ]:
!cd generation && zip -qr sorties_etape1.zip sorties
from google.colab import files
files.download('generation/sorties_etape1.zip')